# Stage 2 V5: Unified Multi-Task Instruction Tuning

## Pourquoi cette version va réussir là où les autres ont échoué

Après analyse des versions V1 à V4, nous avons identifié que le modèle souffrait d'un conflit de format entre la classification pure (Stage 1) et le dialogue (Stage 2). 

### Stratégie V5 :
1. **Format de Réponse Unifié** : Au lieu de demander soit la classe, soit une explication, nous entraînons le modèle à répondre systématiquement selon une structure fixe : 
   `Diagnostic : [CLASSE]. Explication : [CONSEILS EN FRANÇAIS].` 
   Cela force le modèle à effectuer la classification *avant* de générer le texte.
2. **Data Mixing 50/50** : Nous mélangeons les données de classification pure (converties au nouveau format) et les données VQA complexes.
3. **Full Linear LoRA** : Nous étendons le LoRA à tous les modules linéaires (`q, k, v, o, gate, up, down`) pour capturer la complexité du langage français sans dégrader la vision.
4. **Technique Robuste** : Utilisation du `QwenDataCollator` personnalisé qui a fait ses preuves dans vos versions précédentes.

In [8]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os
import json
import re
import random
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from datasets import Dataset
from tqdm import tqdm
from collections import Counter, defaultdict
from unsloth import FastVisionModel
from transformers import AutoProcessor, TrainingArguments, Trainer, EarlyStoppingCallback

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu128
CUDA available: True


In [9]:
# ============================================================
# CELL 2: CONFIGURATION V5
# ============================================================
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
STAGE1_DATA_PATH = BASE_DIR / "data" / "stage1" / "cassava_train_balanced.json"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_v5_unified"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]

TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-5,
    "num_train_epochs": 2,
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 100,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

print("Configuration V5 chargée.")

Configuration V5 chargée.


In [10]:
# ============================================================
# CELL 3: CHARGEMENT DU MODÈLE AVEC LORA ÉTENDU
# ============================================================
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth"
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
print("Modèle chargé avec LoRA étendu (tous les modules linéaires).")

==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Modèle chargé avec LoRA étendu (tous les modules linéaires).


In [11]:
# ============================================================
# CELL 4: PRÉPARATION DES DONNÉES UNIFIÉES (MULTI-TASK)
# ============================================================
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def prepare_unified_dataset():
    s1_data = load_json(STAGE1_DATA_PATH)
    s2_data = load_json(STAGE2_DATA_PATH)
    
    unified = []
    
    # 1. Convertir Stage 1 (Classification) au format unifié
    # On prend un échantillon représentatif pour le 50/50
    random.shuffle(s1_data)
    s1_subset = s1_data[:len(s2_data)] 
    
    for item in s1_subset:
        # Extraction de la classe depuis class_name (Stage 1)
        class_full = item['metadata'].get('class_name', 'Unknown')
        # On transforme la réponse courte en réponse structurée
        item['conversations'][1]['value'] = f"Diagnostic : {class_full}. Cette feuille de manioc présente des symptômes caractéristiques de cette condition."
        item['metadata']['task'] = 'unified_classification'
        item['metadata']['class'] = class_full # Unification de la clé
        unified.append(item)
        
    # 2. Ajouter Stage 2 (VQA)
    for item in s2_data:
        item['metadata']['task'] = 'vqa_dialogue'
        # S'assurer que la clé 'class' existe (déjà le cas en Stage 2)
        unified.append(item)
        
    random.shuffle(unified)
    return unified

mixed_data = prepare_unified_dataset()
print(f"Total samples mixés (50/50): {len(mixed_data)}")
print(f"Distribution des tâches: {Counter(s['metadata']['task'] for s in mixed_data)}")

Total samples mixés (50/50): 58552
Distribution des tâches: Counter({'unified_classification': 29276, 'vqa_dialogue': 29276})


In [12]:
# ============================================================
# CELL 5: FORMATTAGE POUR LE COLLATOR
# ============================================================
def format_sample_for_training(sample):
    """Formate un sample pour l'entraînement."""
    conversations = sample['conversations']
    
    image_path = None
    user_text = ""
    assistant_text = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            match = re.search(r'<img>(.*?)</img>', turn['value'])
            if match:
                image_path = match.group(1)
            user_text = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            assistant_text = turn['value']
    
    return {
        "conversations": [
            {"role": "user", "content": f"Picture 1: <image>\n{user_text}"},
            {"role": "assistant", "content": assistant_text}
        ],
        "image_path": image_path,
        "id": sample['id'],
        "task": sample['metadata'].get('task', 'unknown'),
        "class_name": sample['metadata'].get('class', 'Unknown')
    }

print("Formatting data...")
all_formatted = [format_sample_for_training(s) for s in tqdm(mixed_data, desc="Formatting")]

# Filtrer images valides
valid_samples = [s for s in all_formatted if s['image_path'] and os.path.exists(s['image_path'])]
print(f"\n✅ Valid samples: {len(valid_samples)} / {len(all_formatted)}")

# Split train/val (95/5)
random.seed(42)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * 0.95)
train_dataset_list = valid_samples[:split_idx]
val_dataset_list = valid_samples[split_idx:]

train_dataset = Dataset.from_list(train_dataset_list)
eval_dataset = Dataset.from_list(val_dataset_list)
print(f"Train: {len(train_dataset)}, Val: {len(eval_dataset)}")

Formatting data...


Formatting: 100%|██████████| 58552/58552 [00:00<00:00, 430178.24it/s]


✅ Valid samples: 58552 / 58552


Train: 55624, Val: 2928


In [13]:
# ============================================================
# CELL 6: DATA COLLATOR (Logique technique validée)
# ============================================================
@dataclass
class QwenDataCollator:
    processor: Any
    max_length: int = 1536

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            try:
                image = Image.open(image_path).convert('RGB')
                image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
            except:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            
            batch_images.append([image])
            
            conversations = feature['conversations']
            messages = []
            for turn in conversations:
                role = turn['role']
                content = turn['content']
                if role == 'user' and len(messages) == 0:
                    text_content = content.replace('<image>', '').replace('Picture 1:', '').strip()
                    messages.append({
                        "role": role,
                        "content": [{"type": "image", "image": image}, {"type": "text", "text": text_content}]
                    })
                else:
                    messages.append({"role": role, "content": content.replace('<image>', '').strip()})
            batch_messages.append(messages)

        batch_texts_full = [self.processor.tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch_messages]
        batch_texts_user = [self.processor.tokenizer.apply_chat_template([m for m in ms if m['role'] == 'user'], tokenize=False, add_generation_prompt=True) for ms in batch_messages]

        batch = self.processor(text=batch_texts_full, images=batch_images, return_tensors="pt", padding=True, truncation=True, max_length=self.max_length)
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_enc = self.processor(text=[batch_texts_user[i]], images=[batch_images[i]], return_tensors="pt", padding=False, truncation=True, max_length=self.max_length)
            user_len = user_enc["input_ids"].shape[1]
            if user_len < labels[i].shape[0]:
                labels[i, :user_len] = -100
            else:
                labels[i, :-20] = -100
        
        batch["labels"] = labels
        return batch

data_collator = QwenDataCollator(processor=processor)
print("✅ Data collator technique créé.")

✅ Data collator technique créé.


In [ ]:
# ============================================================
# CELL 7: TRAINER V5
# ============================================================
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🆕 Starting fresh training")
trainer.train()

Lancement de l'entraînement V5...
📌 Resuming from checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified/checkpoint-4800


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55,624 | Num Epochs = 2 | Total steps = 6,954
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 95,178,752 of 8,387,345,408 (1.13% trained)


Step,Training Loss,Validation Loss
4900,1.017400,2.179051
5000,0.992200,2.178571
5100,1.019200,2.178477
5200,1.085500,2.177993
5300,1.034700,2.177919
5400,1.005500,2.177253
5500,1.187800,2.177855
5600,1.019000,2.177704
5700,1.128700,2.177935


Unsloth: Not an error, but Qwen2_5_VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [15]:
# ============================================================
# CELL 8: SAUVEGARDE FINALE
# ============================================================
final_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Modèle sauvegardé dans {final_path}")

Modèle sauvegardé dans /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v5_unified/final_model
